# 🛒 고객 문의 유형 분류기
**유형:** 분류형 (임베딩 + 로지스틱 회귀)  
**목표:** 고객 문의 내용을 입력하면 어떤 유형(AS, 결제, 교환, 반품, 배송, 업무처리, 주문)인지 예측하는 AI 만들기

---
### 데이터 출처
- **K쇼핑 콜센터 질의응답 데이터** (AI Hub 민원 콜센터 데이터셋)
- 7개 카테고리: AS, 결제, 교환, 반품, 배송, 업무처리, 주문
- 고객 질문만 추출하여 분류기 학습

### 이 노트북의 구조
| 단계 | 내용 |
|------|------|
| Step 1 | 데이터 로드 |
| Step 2 | 🧹 데이터 전처리 (핵심!) |
| Step 3 | 임베딩 & 분류기 학습 |
| Step 4 | 예측 & 평가 |
| Step 5 | 미니 미션 |

### 💡 왜 데이터 전처리가 중요한가?
원본 데이터는 **콜센터 통화 녹취를 텍스트로 변환**한 것이라 다음과 같은 문제가 있습니다:
- `"네."`, `"아 그래요?"` 같은 **의미 없는 짧은 응답**이 전체의 3~5%
- 한 통화의 고객 발화가 **문장 단위로 쪼개져** 있어 맥락 없는 단편만 존재
- `"ㅇㅇㅇ"` 같은 **개인정보 마스킹 토큰**이 포함
- `"아 그래요?"`가 **모든 카테고리에 공통 등장** → 노이즈

**전처리 없이는 AI가 노이즈를 학습하게 됩니다!**

## Step 1: 데이터 로드 (원본 그대로)

In [ ]:
# 1. 필수 라이브러리 설치 & 임포트
!pip install sentence-transformers scikit-learn torch

import json, zipfile, re
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict

In [ ]:
# 2. K쇼핑 콜센터 데이터 로드 (zip → JSON → DataFrame)
# 💡 핵심: 개별 문장이 아니라 "대화셋" 단위로 고객 발화를 결합합니다

train_dir = Path("../Data/1.Training/라벨링데이터_231222_add")
val_dir = Path("../Data/2.Validation/라벨링데이터_231222_add")
categories = ["AS", "결제", "교환", "반품", "배송", "업무처리", "주문"]

def load_kshopping_data(base_dir, split_name):
    """K쇼핑 zip 파일에서 대화셋 단위로 고객 질문을 결합하여 추출"""
    all_dialogs = []
    
    for cat in categories:
        zip_name = f"민원(콜센터) 질의응답_K쇼핑_{cat}_{split_name}.zip"
        zip_path = base_dir / zip_name
        
        if not zip_path.exists():
            print(f"⚠️ 파일 없음: {zip_name}")
            continue
        
        with zipfile.ZipFile(zip_path, 'r') as zf:
            for fname in zf.namelist():
                with zf.open(fname) as f:
                    data = json.load(f)
        
        # 대화셋 단위로 그룹핑
        dialog_groups = defaultdict(list)
        for d in data:
            dialog_groups[d["대화셋일련번호"]].append(d)
        
        # 각 대화셋에서 고객 질문만 결합
        for dialog_id, turns in dialog_groups.items():
            customer_texts = [
                d["고객질문(요청)"].strip()
                for d in turns
                if d.get("QA") == "Q" and d.get("고객질문(요청)", "").strip()
            ]
            if customer_texts:
                combined = " ".join(customer_texts)
                all_dialogs.append({
                    "text": combined,
                    "category": cat,
                    "dialog_id": dialog_id,
                    "num_turns": len(customer_texts)
                })
        
        print(f"  ✅ {cat}: {len([d for d in all_dialogs if d['category']==cat])}건 (대화셋 단위)")
    
    return pd.DataFrame(all_dialogs)

print("🔹 Training 데이터 로드 중...")
train_raw_df = load_kshopping_data(train_dir, "Training")
print(f"\n📊 Training 전체 (대화셋 단위): {len(train_raw_df)}건")

print("\n🔹 Validation 데이터 로드 중...")
val_raw_df = load_kshopping_data(val_dir, "Validation")
print(f"\n📊 Validation 전체 (대화셋 단위): {len(val_raw_df)}건")

In [ ]:
# 3. 원본 데이터 품질 확인 (전처리 전)
print("=== 📊 원본 데이터 품질 분석 ===\n")

# 길이 분포
lengths = train_raw_df["text"].str.len()
print(f"텍스트 길이 통계:")
print(f"  평균: {lengths.mean():.0f}자, 중앙값: {lengths.median():.0f}자")
print(f"  최소: {lengths.min()}자, 최대: {lengths.max()}자")

# 짧은 텍스트 분석
short_mask = lengths < 10
print(f"\n10자 미만 텍스트: {short_mask.sum()}건 ({short_mask.mean()*100:.1f}%)")
if short_mask.sum() > 0:
    print("  예시:", train_raw_df[short_mask]["text"].head(5).tolist())

# ㅇㅇㅇ 포함
anon_mask = train_raw_df["text"].str.contains("ㅇㅇ")
print(f"\nㅇㅇㅇ(익명화) 포함: {anon_mask.sum()}건 ({anon_mask.mean()*100:.1f}%)")

# 카테고리 무관 공통 표현
from collections import Counter
all_texts = train_raw_df["text"].tolist()
text_counter = Counter(all_texts)
print(f"\n가장 빈번한 텍스트 (노이즈 후보):")
for text, count in text_counter.most_common(5):
    print(f"  [{count}회] {text[:50]}...")

print(f"\n=== 카테고리별 분포 ===")
print(train_raw_df["category"].value_counts().sort_index())

---
## Step 2: 🧹 데이터 전처리

위 분석에서 발견한 문제점을 해결합니다:

| 문제 | 해결 방법 |
|------|----------|
| 의미 없는 짧은 텍스트 | 최소 글자 수 필터링 (15자 이상) |
| ㅇㅇㅇ 익명화 토큰 | 정규식으로 제거 |
| 불필요한 공백/특수문자 | 정규화 처리 |
| "아 그래요?" 같은 필러 표현 | 필러 패턴 제거 |
| 중복 텍스트 | 중복 제거 (deduplicate) |
| 카테고리 불균형 | 균등 샘플링 |

In [ ]:
# 4. 전처리 함수 정의 & 적용

def clean_text(text):
    """텍스트 정제: 노이즈 제거 & 정규화"""
    # 1) ㅇㅇㅇ 익명화 토큰 제거
    text = re.sub(r'ㅇ{2,}', '', text)
    
    # 2) 필러(filler) 표현 제거 — 분류에 도움이 안 되는 반응형 표현
    filler_patterns = [
        r'\b아\s*그래요\??',       # 아 그래요?
        r'\b네[\.\s]*$',           # 네. / 네
        r'\b예[\.\s]*$',           # 예.
        r'\b여보세요\??',          # 여보세요?
        r'\b아\s*네[\.\s]*',       # 아 네.
        r'\b잠깐만요[\.\s]*',      # 잠깐만요.
        r'\b감사합니다[\.\s]*',    # 감사합니다.
        r'\b수고하십니다[\.\s]*',  # 수고하십니다.
        r'\b안녕하세요[\.\?\s]*',  # 안녕하세요?
    ]
    for pattern in filler_patterns:
        text = re.sub(pattern, ' ', text)
    
    # 3) 연속 공백 정리
    text = re.sub(r'\s+', ' ', text).strip()
    
    # 4) 양쪽 구두점 정리
    text = text.strip('.,!? ')
    
    return text

def preprocess_dataframe(df, min_length=15):
    """DataFrame에 전처리 파이프라인 적용"""
    original_count = len(df)
    
    # Step 1: 텍스트 정제
    df = df.copy()
    df["text_clean"] = df["text"].apply(clean_text)
    
    # Step 2: 빈 텍스트 제거
    df = df[df["text_clean"].str.len() > 0]
    after_empty = len(df)
    
    # Step 3: 최소 길이 필터링 (너무 짧으면 분류 단서 부족)
    df = df[df["text_clean"].str.len() >= min_length]
    after_length = len(df)
    
    # Step 4: 완전 중복 제거
    df = df.drop_duplicates(subset=["text_clean"])
    after_dedup = len(df)
    
    # 원본 text를 정제된 text로 교체
    df["text"] = df["text_clean"]
    df = df.drop(columns=["text_clean"])
    
    print(f"  원본: {original_count}건")
    print(f"  → 빈 텍스트 제거 후: {after_empty}건 (-{original_count - after_empty})")
    print(f"  → {min_length}자 미만 제거 후: {after_length}건 (-{after_empty - after_length})")
    print(f"  → 중복 제거 후: {after_dedup}건 (-{after_length - after_dedup})")
    
    return df.reset_index(drop=True)

# 전처리 적용
print("🧹 Training 데이터 전처리 중...")
train_clean_df = preprocess_dataframe(train_raw_df)

print("\n🧹 Validation 데이터 전처리 중...")
val_clean_df = preprocess_dataframe(val_raw_df)

In [ ]:
# 5. 전처리 전 vs 후 비교
print("=== 전처리 전 vs 후 비교 ===\n")

# 전처리 전 예시
print("📌 전처리 전 (원본):")
for i, row in train_raw_df.head(3).iterrows():
    print(f"  [{row['category']}] {row['text'][:80]}...")

print(f"\n📌 전처리 후 (정제):")
for i, row in train_clean_df.head(3).iterrows():
    print(f"  [{row['category']}] {row['text'][:80]}...")

# 길이 분포 비교
print(f"\n📊 텍스트 길이 비교:")
print(f"  전처리 전 — 평균: {train_raw_df['text'].str.len().mean():.0f}자, 중앙값: {train_raw_df['text'].str.len().median():.0f}자")
print(f"  전처리 후 — 평균: {train_clean_df['text'].str.len().mean():.0f}자, 중앙값: {train_clean_df['text'].str.len().median():.0f}자")

# 카테고리 분포
print(f"\n📊 전처리 후 카테고리 분포:")
print(train_clean_df["category"].value_counts().sort_index())

In [ ]:
# 6. 균등 샘플링 (시간 절약 + 카테고리 균형)
SAMPLE_PER_CAT = 500

train_df = train_clean_df.groupby("category").apply(
    lambda x: x.sample(n=min(len(x), SAMPLE_PER_CAT), random_state=42)
).reset_index(drop=True)

test_df = val_clean_df.groupby("category").apply(
    lambda x: x.sample(n=min(len(x), SAMPLE_PER_CAT // 5), random_state=42)
).reset_index(drop=True)

print(f"최종 train: {len(train_df)}건")
print(f"최종 test: {len(test_df)}건")
print(f"\n=== Train 카테고리별 분포 ===")
print(train_df["category"].value_counts().sort_index())
print(f"\n=== Test 카테고리별 분포 ===")
print(test_df["category"].value_counts().sort_index())

# 샘플 확인
print("\n=== 전처리된 질문 예시 ===")
for cat in categories:
    sample = train_df[train_df["category"] == cat]["text"].iloc[0]
    print(f"\n🏷️ [{cat}]")
    print(f"   → {sample[:100]}{'...' if len(sample)>100 else ''}")

In [ ]:
# 7. 라벨 인코딩
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
train_df["label"] = label_encoder.fit_transform(train_df["category"])
test_df["label"]  = label_encoder.transform(test_df["category"])

print("카테고리 → 숫자 매핑:")
for i, cls in enumerate(label_encoder.classes_):
    print(f"  {i}: {cls}")

---
## Step 3: 임베딩 & 분류기 학습

In [ ]:
# 8. 문장 임베딩
# ⏳ 이 셀은 3~5분 걸립니다!
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

def embed_texts(texts, batch_size=64):
    return model.encode(texts, batch_size=batch_size, show_progress_bar=True, convert_to_numpy=True)

print("🔹 train 임베딩 중...")
X_train = embed_texts(train_df["text"].tolist())
y_train = train_df["label"].values

print("🔹 test 임베딩 중...")
X_test = embed_texts(test_df["text"].tolist())
y_test = test_df["label"].values

print(f"\n임베딩 shape: train {X_train.shape}, test {X_test.shape}")

In [ ]:
# 9. 로지스틱 회귀 분류기 학습
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

clf = LogisticRegression(max_iter=2000)

print("🔹 고객 문의 분류기 학습 중...")
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"\n✅ Test Accuracy: {acc:.4f}")

print("\n=== 분류 리포트 ===")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

---
## Step 4: 예측 & 평가

In [ ]:
# 10. 예측 함수 (입력도 전처리 적용)
def predict_inquiry(text: str, top_k: int = 3):
    """고객 문의 텍스트를 입력받아 유형을 예측 (전처리 포함)"""
    cleaned = clean_text(text)  # 동일한 전처리 적용
    emb = model.encode([cleaned], convert_to_numpy=True)
    probs = clf.predict_proba(emb)[0]
    top_indices = np.argsort(-probs)[:top_k]

    print("\n" + "=" * 50)
    print("📞 고객 문의 내용:")
    print(f"   {text}")
    if cleaned != text:
        print(f"   (전처리 후: {cleaned[:60]}{'...' if len(cleaned)>60 else ''})")
    print("=" * 50)
    print("\n🔮 예측된 문의 유형 (Top-k):")
    for idx in top_indices:
        cat_name = label_encoder.classes_[idx]
        p = probs[idx]
        bar = "█" * int(p * 20)
        print(f"   {cat_name:8s} {bar} ({p:.3f})")
    print("=" * 50 + "\n")

In [ ]:
# 11. 테스트: 명확한 문의
print(">>> 테스트 1: 배송 문의")
predict_inquiry("주문한 물건이 아직 안 왔어요. 배송 조회 좀 해주세요.")

print(">>> 테스트 2: 반품 문의")
predict_inquiry("이 제품 환불하고 싶어요. 반품 접수해주세요.")

print(">>> 테스트 3: 결제 문의")
predict_inquiry("카드 결제가 안 돼요. 다른 결제 수단으로 바꿀 수 있나요?")

---
## Step 5: 미니 미션 🎯

### 🎯 미션 1: 직접 고객 문의 작성하기
각 카테고리에 맞는 고객 문의를 직접 작성하고, AI가 올바르게 분류하는지 확인하세요.

In [ ]:
# 미션 1-1: AS 관련 문의
my_inquiry_1 = "TV 화면이 깨져서 나와요. AS 접수하려면 어떻게 해야 하나요?"
predict_inquiry(my_inquiry_1)
# 👉 예상한 카테고리: AS
# 👉 AI의 예측이 맞았나요?: 

In [ ]:
# 미션 1-2: 교환 관련 문의
my_inquiry_2 = "사이즈가 안 맞아서 교환하고 싶어요. 다른 사이즈로 바꿀 수 있나요?"
predict_inquiry(my_inquiry_2)
# 👉 예상한 카테고리: 교환
# 👉 AI의 예측이 맞았나요?: 

In [ ]:
# 미션 1-3: 주문 관련 문의
my_inquiry_3 = "주문을 취소하고 싶은데요. 아직 출발 전이면 취소되나요?"
predict_inquiry(my_inquiry_3)
# 👉 예상한 카테고리: 주문
# 👉 AI의 예측이 맞았나요?: 

### 🎯 미션 2: 경계가 애매한 문의 만들기
하나의 문의가 **두 가지 카테고리에 동시에 해당**할 수 있는 문장을 만들어보세요.  
예: "주문한 물건이 불량이라 교환하고 싶어요" → 주문? 교환? AS?

In [ ]:
# 미션 2-1: 교환 + 반품이 애매한 문의
ambiguous_1 = "상품이 마음에 안 들어서 다른 걸로 바꾸거나 그냥 돌려보내고 싶어요."
predict_inquiry(ambiguous_1)
# 👉 해당할 수 있는 카테고리 2개: 
# 👉 AI 예측이 적절한가요?: 

In [ ]:
# 미션 2-2: 결제 + 주문이 애매한 문의
ambiguous_2 = "주문하면서 결제했는데 금액이 이상해요. 확인 좀 해주세요."
predict_inquiry(ambiguous_2)
# 👉 해당할 수 있는 카테고리 2개: 
# 👉 AI 예측이 적절한가요?: 

In [ ]:
# 미션 2-3: 배송 + 반품이 애매한 문의
ambiguous_3 = "배송 온 물건이 파손되어 있어서 반품하려고요. 택배 다시 보내야 하나요?"
predict_inquiry(ambiguous_3)
# 👉 해당할 수 있는 카테고리 2개: 
# 👉 AI 예측이 적절한가요?: 

### 🎯 미션 3: AI 확신도 분석
`predict_inquiry`는 상위 3개 카테고리와 확률을 보여줍니다.

- 1위 확률이 0.8 이상 → AI가 **확신**하는 것
- 1위와 2위 확률이 비슷 → AI가 **헷갈리는** 것

이 두 가지 경우를 각각 찾아보세요.

In [ ]:
# 미션 3-1: AI가 확신하는 문의 (1위 확률 0.8 이상 목표)
certain_inquiry = "배송이 언제 되나요? 송장번호 알려주세요."
predict_inquiry(certain_inquiry)
# 👉 1위 확률: 

In [ ]:
# 미션 3-2: AI가 헷갈리는 문의 (1위와 2위 확률 차이 0.1 이하 목표)
uncertain_inquiry = "이 물건 문제가 있는데 바꿔주실 수 있나요? 아니면 그냥 돈 돌려받을게요."
predict_inquiry(uncertain_inquiry)
# 👉 1위와 2위의 확률 차이: 

### 🎯 미션 4: 실제 업무 활용 아이디어
이 분류기를 실제 쇼핑몰 고객센터에 적용한다면 어떻게 활용할 수 있을까요?

In [ ]:
# ✏️ 활용 아이디어를 적어보세요 (주석으로)
#
# 1. 이 분류기를 실제 고객센터에 적용하면 어떤 점이 좋을까요?
#    → 
#
# 2. 분류 정확도를 높이려면 어떤 방법이 있을까요? (2가지 이상)
#    → 
#
# 3. 7개 카테고리 외에 추가하면 좋을 카테고리가 있다면?
#    → 
#
# 4. 이 분류기의 한계점은 무엇일까요?
#    → 